# Shaxsiy AI Platforma - Video Generator
Bu notebook orqali matndan video yaratish (Text-to-Video) backend logikasini sinab ko'rishingiz mumkin.

In [ ]:
# 1. Kerakli kutubxonalarni yuklab olamiz
!pip install -q diffusers transformers accelerate mediapy
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Google Drive-ni ulash (Tayyor videoni saqlash uchun)
from google.colab import drive
import os

print("Google Drive ulanmoqda...")
drive.mount('/content/drive')

# Videolar saqlanadigan papkani yaratamiz
save_path = "/content/drive/MyDrive/AI_Platform_Videos"
os.makedirs(save_path, exist_ok=True)
print(f"Tayyor videolar shu yerga saqlanadi: {save_path}")

In [ ]:
import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

# Colab T4 GPU-dan unumli foydalanish uchun CogVideoX-2b modelini tanlaymiz
model_id = "THUDM/cogvideox-2b"

print("AI Modeli xotiraga yuklanmoqda (Bu bir oz vaqt olishi mumkin)...")
pipe = CogVideoXPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

# GPU xotirasini tejash va tezlashtirish sozlamalari
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

print("Model muvaffaqiyatli yuklandi va generatsiyaga tayyor!")

In [ ]:
import time

def generate_ai_video(prompt, num_frames=49, guidance_scale=6.0, num_inference_steps=50):
    """
    Matn asosida video yaratuvchi asosiy funksiya.
    Bu funksiyani keyinchalik shaxsiy platformangiz backend-iga (API) oson ko'chirishingiz mumkin.
    """
    print(f"Generatsiya boshlandi. Prompt: '{prompt}'")
    start_time = time.time()
    
    # Videoni yaratish
    video_frames = pipe(
        prompt=prompt,
        num_videos_per_prompt=1,
        num_frames=num_frames,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=torch.manual_seed(42),
    ).frames
    
    # Fayl nomini vaqtga qarab unikal qilish
    timestamp = int(time.time())
    output_filename = f"tayyor_video_{timestamp}.mp4"
    full_output_path = os.path.join(save_path, output_filename)
    
    # Videoni saqlash
    export_to_video(video_frames, full_output_path, fps=8)
    
    end_time = time.time()
    print(f"Muvaffaqiyatli yakunlandi! Sarflangan vaqt: {int(end_time - start_time)} soniya.")
    print(f"Video saqlangan joy: {full_output_path}")
    
    return full_output_path

In [ ]:
# O'zingiz xohlagan inglizcha promptni yozing
user_prompt = "A cinematic shot of a futuristic neon city with flying cars, cyber punk style, 4k resolution"

# Funksiyani ishga tushiramiz
video_fayl_yo_li = generate_ai_video(prompt=user_prompt)